# Check: projected Zeeman matrices $M_\alpha \propto \sigma_\alpha$ and signs of $g_{ii}$

Top valence Kramers pair of the 2H-Ge nanowire (wire axis $x$, cross-section $y$-$z$), at **one electric-field strength, $|\mathcal{E}| = 6$ V/µm**, for several field directions in the $y$-$z$ plane.

**Method**

1. Build $H_0$ (B = 0) with kwant, add the electrostatic potential $V = -\mathcal{E}\cdot(\mathbf r - L/2)$, and get the Kramers pair $P = (|1\rangle, |2\rangle)$ by shift-invert `eigsh`.
2. Fix **one** pseudospin basis for all three field directions (`psi_new_basis`: $S_z$ diagonal in the pair, $\langle 1|S_x|2\rangle$ real and positive), i.e. the same convention as `EDSR/M_matrix/export_M*.py`.
3. For $\alpha = x, y, z$:
   $$M_\alpha = P^\dagger \frac{\partial H}{\partial B_\alpha} P = \frac{\mu_B}{2}\sum_\beta g_{\alpha\beta}\,\sigma_\beta, \qquad g_{\alpha\beta} = \frac{\mathrm{Tr}(M_\alpha \sigma_\beta)}{\mu_B},$$
   where $\partial H/\partial B_\alpha$ = orbital part (central difference, Peierls / $k_x$ shift) + spin part $g_s\mu_B S_\alpha$.
   - $B_x$: $\mathbf A = (0,0,B y)$, Peierls phases on the $y$-$z$ hoppings
   - $B_y$: $\mathbf A = (B z,0,0)$, $k_x \to k_x + eBz/\hbar$ at the site / bond midpoint
   - $B_z$: $\mathbf A = (-B y,0,0)$, $k_x \to k_x - eBy/\hbar$ at the site / bond midpoint
4. Checks:
   - **$M_\alpha \propto \sigma_\alpha$ in this basis**: the off-diagonal elements $g_{\alpha\beta}$ ($\beta\neq\alpha$) should be $\ll |g_{\alpha\alpha}|$, and $\mathrm{Tr}\,M_\alpha = 0$.
   - **Basis-independent version**: the $g$-vectors $\mathbf g_\alpha = (g_{\alpha x}, g_{\alpha y}, g_{\alpha z})$ are mutually orthogonal, i.e. $G G^T$ is diagonal, so $x, y, z$ are principal axes and *some* pseudospin basis makes every $M_\alpha \propto \sigma_\alpha$.
   - **Signs**: individual signs of $g_{ii}$ depend on the pseudospin convention (a $\pi$ rotation of the pseudospin flips two of them). The invariant is $\operatorname{sign}\det G$ (= $\operatorname{sign}(g_{xx}g_{yy}g_{zz})$ when $G$ is diagonal). Both are recorded.


In [1]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "4")

import time
import numpy as np
import kwant
from scipy.sparse.linalg import eigsh
from Hamiltonian_mathematica_v2 import gso, spin_matrix_small, apply_spin_operator, psi_new_basis

np.set_printoptions(precision=5, suppress=True, linewidth=120)

In [2]:
# ---------------- parameters ----------------
N = int(os.environ.get("GF_N", 100))   # sites per direction (N = 100 matches EDSR/import_data)
Ny = Nz = N
L = 300.0             # Angstrom
Nband = 10
kx = 0.0
Bf = 0.01             # T, finite-difference step (orbital term)
muB = 5.788e-5        # eV/T
gspin = 2.0
hbar = 1.05457e-34
h = 6.626e-34
q = 1.602e-19
sigma_val = 0.12      # eV, shift-invert target
N_EIG = 4             # Kramers pair + next pair (gap check)

E_V_um = 6.0          # |E| (V/um)
# direction: theta (deg) measured from +z toward +y  (0 -> E_z, 90 -> E_y)
THETAS_DEG = [90.0, 0.0, 83.2]

a = L / (Ny + 1)
SX, SY, SZ = spin_matrix_small()
S_SPIN = {"x": SX, "y": SY, "z": SZ}
PAULI = {"x": np.array([[0, 1], [1, 0]], complex),
         "y": np.array([[0, -1j], [1j, 0]], complex),
         "z": np.array([[1, 0], [0, -1]], complex)}
AX = ["x", "y", "z"]
print(f"N = {N}, a = {a:.3f} A, Hilbert-space dim = {N*N*Nband}")

N = 100, a = 2.970 A, Hilbert-space dim = 100000


## Orbital Hamiltonians for $\pm B_\alpha$

$k_x$ shift for $B_y$, $B_z$ evaluated at the site (onsite) or the bond midpoint (hopping), uniformly for every block. This is the corrected convention of `EDSR/M_matrix/Hamiltonian_mathematica_v2.py`. It is equivalent to `gs_shifted_by/bz` in this folder called with the tail-site coordinate.

In [3]:
def kx_shift(B, n):
    # e B (n a) / hbar  in 1/Angstrom  (n in lattice units)
    return 2 * np.pi / a * n * B * (a * 1e-10) ** 2 / (h / q)


def make_system(field, B):
    yc = (Ny - 1) / 2.0
    zc = (Nz - 1) / 2.0
    blocks0 = {(dy, dz): np.asarray(gso(a, kx, dy, dz), complex)
               for dy in (-1, 0, 1) for dz in (-1, 0, 1)}
    pref = q * B * (a * 1e-10) ** 2 / hbar

    def onsite(site):
        y, z = site.tag
        if field == "y" and B != 0:
            return np.asarray(gso(a, kx + kx_shift(B, z - zc), 0, 0), complex)
        if field == "z" and B != 0:
            return np.asarray(gso(a, kx - kx_shift(B, y - yc), 0, 0), complex)
        return blocks0[(0, 0)]

    def hopping(s1, s2):
        y1, z1 = s1.tag
        y2, z2 = s2.tag
        dy, dz = int(y1 - y2), int(z1 - z2)
        if field == "x" and B != 0:          # A = (0, 0, B y): Peierls phase
            ymid = 0.5 * (y1 + y2) - yc
            return blocks0[(dy, dz)] * np.exp(-1j * pref * ymid * dz)
        if field == "y" and B != 0:          # A = (B z, 0, 0)
            zmid = 0.5 * (z1 + z2) - zc
            return np.asarray(gso(a, kx + kx_shift(B, zmid), dy, dz), complex)
        if field == "z" and B != 0:          # A = (-B y, 0, 0)
            ymid = 0.5 * (y1 + y2) - yc
            return np.asarray(gso(a, kx - kx_shift(B, ymid), dy, dz), complex)
        return blocks0[(dy, dz)]

    syst = kwant.Builder()
    lat = kwant.lattice.square(norbs=Nband)
    syst[(lat(y, z) for y in range(Ny) for z in range(Nz))] = onsite
    for d in [(1, 0), (0, 1), (1, 1), (1, -1)]:
        syst[kwant.builder.HoppingKind(d, lat, lat)] = hopping
    return syst.finalized()


def tags_of(fs):
    return np.array([s.tag for s in fs.sites], float)


t0 = time.time()
f0 = make_system(None, 0.0)
TAGS = tags_of(f0)
H0 = f0.hamiltonian_submatrix(sparse=True).tocsr()
dH_orb = {}
for ax in AX:
    fp, fm = make_system(ax, +Bf), make_system(ax, -Bf)
    assert np.array_equal(tags_of(fp), TAGS) and np.array_equal(tags_of(fm), TAGS)
    dH_orb[ax] = ((fp.hamiltonian_submatrix(sparse=True)
                   - fm.hamiltonian_submatrix(sparse=True)) / (2 * Bf)).tocsr()
    print(f"dH/dB_{ax} built,  ||dH - dH^dag|| = {abs(dH_orb[ax] - dH_orb[ax].getH()).max():.1e}")
print(f"systems built in {time.time() - t0:.0f} s")

dH/dB_x built,  ||dH - dH^dag|| = 0.0e+00


dH/dB_y built,  ||dH - dH^dag|| = 0.0e+00


dH/dB_z built,  ||dH - dH^dag|| = 0.0e+00
systems built in 15 s


## Kramers pair, common pseudospin basis, $M_x, M_y, M_z$

In [4]:
def potential_diagonal(Ef, theta):
    ypos = (TAGS[:, 0] + 1) * a - L / 2
    zpos = (TAGS[:, 1] + 1) * a - L / 2
    return np.repeat(-Ef * (np.sin(theta) * ypos + np.cos(theta) * zpos), Nband)


def zeeman_matrices(theta_deg):
    Ef = E_V_um * 1e-4                                   # V/Angstrom
    H = H0.copy()
    H.setdiag(H.diagonal() + potential_diagonal(Ef, np.deg2rad(theta_deg)))
    w, v = eigsh(H, k=N_EIG, sigma=sigma_val, which="LM")
    order = np.argsort(np.abs(w - sigma_val))
    pair, rest = order[:2], order[2:]
    info = dict(E_pair=w[pair].mean(), split0=abs(w[pair[0]] - w[pair[1]]),
                gap_next=np.min(np.abs(w[rest][:, None] - w[pair][None, :])))
    P, _ = np.linalg.qr(v[:, pair])                      # orthonormalize inside the pair
    p1, p2 = psi_new_basis(P[:, 0], P[:, 1], None)       # ONE basis for x, y, z
    P = np.column_stack((p1, p2))
    info["Sz_pair"] = [np.vdot(p, apply_spin_operator(p, SZ)).real for p in (p1, p2)]

    M = {}
    for ax in AX:
        DP = np.column_stack([dH_orb[ax] @ P[:, j] + gspin * muB * apply_spin_operator(P[:, j], S_SPIN[ax])
                              for j in range(2)])
        Mraw = P.conj().T @ DP
        info[f"herm_err_{ax}"] = np.abs(Mraw - Mraw.conj().T).max() / np.abs(Mraw).max()
        M[ax] = 0.5 * (Mraw + Mraw.conj().T)             # eV/T
    return M, info


def g_tensor(M):
    # G[alpha, beta] = Tr(M_alpha sigma_beta) / muB
    return np.array([[np.trace(M[al] @ PAULI[be]).real / muB for be in AX] for al in AX])


results = {}
for th in THETAS_DEG:
    t0 = time.time()
    M, info = zeeman_matrices(th)
    results[th] = (M, g_tensor(M), info)
    print(f"theta = {th:5.1f} deg done ({time.time() - t0:.0f} s): E_pair = {info['E_pair']:.6f} eV, "
          f"Kramers splitting = {info['split0']:.1e} eV, gap to next = {info['gap_next']*1e3:.3f} meV, "
          f"<S_z> = {np.round(info['Sz_pair'], 4)}")

theta =  90.0 deg done (32 s): E_pair = 0.034326 eV, Kramers splitting = 4.7e-16 eV, gap to next = 1.457 meV, <S_z> = [-0.3233  0.3233]


theta =   0.0 deg done (28 s): E_pair = 0.053280 eV, Kramers splitting = 2.1e-16 eV, gap to next = 11.974 meV, <S_z> = [-0.4995  0.4995]


theta =  83.2 deg done (31 s): E_pair = 0.038196 eV, Kramers splitting = 9.7e-17 eV, gap to next = 4.802 meV, <S_z> = [-0.4673  0.4673]


## Check 1: is $M_\alpha \propto \sigma_\alpha$?

For every $M_\alpha$ we print the matrix, its decomposition $M_\alpha/\mu_B = \tfrac12(c_0\mathbb 1 + \sum_\beta g_{\alpha\beta}\sigma_\beta)$, the reconstruction error, and the leakage $\max_{\beta\ne\alpha}|g_{\alpha\beta}|/|\mathbf g_\alpha|$.

In [5]:
TOL = 1e-2   # leakage tolerance (fraction of |g_alpha|)

def report(th):
    M, G, info = results[th]
    lab = {90.0: "E || y", 0.0: "E || z"}.get(th, f"theta = {th} deg")
    print("=" * 90)
    print(f"|E| = {E_V_um} V/um,  {lab}  (theta = {th} deg from +z toward +y)")
    print("=" * 90)
    ok_all = True
    for i, al in enumerate(AX):
        Mal = M[al] / muB
        c0 = np.trace(Mal).real
        recon = 0.5 * (c0 * np.eye(2) + sum(G[i, j] * PAULI[be] for j, be in enumerate(AX)))
        gnorm = np.linalg.norm(G[i])
        leak = max(abs(G[i, j]) for j in range(3) if j != i) / gnorm
        ok = leak < TOL
        ok_all &= ok
        print(f"\nM_{al}/muB =\n{Mal}")
        print(f"  Tr M_{al}/muB = {c0:+.2e}   recon. error = {np.abs(recon - Mal).max():.1e}   "
              f"(M - M^dag)/|M| before symmetrization = {info['herm_err_' + al]:.1e}")
        print(f"  g_{al}x, g_{al}y, g_{al}z = {G[i, 0]:+.5f}, {G[i, 1]:+.5f}, {G[i, 2]:+.5f}   "
              f"|g_{al}| = {gnorm:.5f}")
        print(f"  leakage max|g_{al}b|/|g_{al}| (b != {al}) = {leak:.2e}  ->  "
              f"M_{al} ∝ sigma_{al}: {'YES' if ok else 'NO'}")
    return ok_all


summary_prop = {th: report(th) for th in THETAS_DEG}

|E| = 6.0 V/um,  E || y  (theta = 90.0 deg from +z toward +y)

M_x/muB =
[[-0.     +0.j -0.09701+0.j]
 [-0.09701-0.j  0.     +0.j]]
  Tr M_x/muB = -2.66e-14   recon. error = 1.3e-29   (M - M^dag)/|M| before symmetrization = 2.4e-15
  g_xx, g_xy, g_xz = -0.19401, -0.00000, -0.00000   |g_x| = 0.19401
  leakage max|g_xb|/|g_x| (b != x) = 5.11e-13  ->  M_x ∝ sigma_x: YES

M_y/muB =
[[-0.+0.j      -0.+1.65996j]
 [-0.-1.65996j  0.+0.j     ]]
  Tr M_y/muB = -3.21e-14   recon. error = 6.3e-30   (M - M^dag)/|M| before symmetrization = 2.7e-15
  g_yx, g_yy, g_yz = -0.00000, -3.31993, -0.00000   |g_y| = 3.31993
  leakage max|g_yb|/|g_y| (b != y) = 3.20e-14  ->  M_y ∝ sigma_y: YES

M_z/muB =
[[ 6.81262+0.j  0.     +0.j]
 [ 0.     -0.j -6.81262+0.j]]
  Tr M_z/muB = -5.99e-13   recon. error = 3.9e-31   (M - M^dag)/|M| before symmetrization = 1.1e-16
  g_zx, g_zy, g_zz = +0.00000, -0.00000, +13.62524   |g_z| = 13.62524
  leakage max|g_zb|/|g_z| (b != z) = 2.42e-15  ->  M_z ∝ sigma_z: YES
|E| = 6.0 V/

## Check 2 (basis-independent): are $x,y,z$ principal axes?

$\cos\angle(\mathbf g_\alpha, \mathbf g_\beta)$ for $\alpha \ne \beta$ should vanish. If it does, the projected matrices are $\propto\sigma_x,\sigma_y,\sigma_z$ in some pseudospin basis, even if the `psi_new_basis` convention does not pick it. Also compared: $|\mathbf g_\alpha|$ (basis-invariant) vs. $|g_{\alpha\alpha}|$.

In [6]:
for th in THETAS_DEG:
    M, G, info = results[th]
    gn = np.linalg.norm(G, axis=1)
    cos = (G @ G.T) / np.outer(gn, gn)
    print(f"theta = {th:5.1f} deg:  cos(g_x,g_y) = {cos[0,1]:+.2e}, cos(g_x,g_z) = {cos[0,2]:+.2e}, "
          f"cos(g_y,g_z) = {cos[1,2]:+.2e}")
    print(f"     |g_x|,|g_y|,|g_z| = {np.round(gn, 5)}   |g_xx|,|g_yy|,|g_zz| = {np.round(np.abs(np.diag(G)), 5)}")

theta =  90.0 deg:  cos(g_x,g_y) = +3.56e-13, cos(g_x,g_z) = -5.13e-13, cos(g_y,g_z) = -3.17e-14
     |g_x|,|g_y|,|g_z| = [ 0.19401  3.31993 13.62524]   |g_xx|,|g_yy|,|g_zz| = [ 0.19401  3.31993 13.62524]
theta =   0.0 deg:  cos(g_x,g_y) = +1.53e-14, cos(g_x,g_z) = -1.37e-15, cos(g_y,g_z) = -3.05e-14
     |g_x|,|g_y|,|g_z| = [ 0.13438  0.36667 11.04379]   |g_xx|,|g_yy|,|g_zz| = [ 0.13438  0.36667 11.04379]
theta =  83.2 deg:  cos(g_x,g_y) = -3.48e-13, cos(g_x,g_z) = +3.66e-13, cos(g_y,g_z) = -9.88e-01
     |g_x|,|g_y|,|g_z| = [ 0.06589  6.60536 17.50714]   |g_xx|,|g_yy|,|g_zz| = [ 0.06589  0.63411 17.47536]


## Check 3: cross-check with the stored matrices

* `EDSR/M_matrix/M*_...npy`: exported by `EDSR/M_matrix/export_M*.py` after commit 30644f9 (Peierls sign fixed to $e^{-i\ldots}$).
* `EDSR/import_data/M*_...npy`: the copies `main_EDSR_sphere_map.py` reads. **The `Mx` files there predate the Peierls-sign fix**, so they are expected to disagree for $M_x$.

Same N = 100, L = 300 Å and basis convention, so a direct comparison is valid. Only runs for N = 100.

In [7]:
if N == 100:
    for sub in ("M_matrix", "import_data"):
        print(f"--- EDSR/{sub} ---")
        for th in THETAS_DEG:
            M = results[th][0]
            for ax in AX:
                fn = os.path.join("..", "..", "EDSR", sub, f"M{ax}_Eth{th:.2f}_{E_V_um:.3f}Vum_N100_L300.npy")
                if os.path.exists(fn):
                    Mst = np.load(fn)
                    print(f"theta = {th:5.1f}  M_{ax}: max|M_new - M_stored|/max|M_stored| = "
                          f"{np.abs(M[ax] - Mst).max() / np.abs(Mst).max():.2e}")
else:
    print("N != 100: skipped")

--- EDSR/M_matrix ---
theta =  90.0  M_x: max|M_new - M_stored|/max|M_stored| = 7.71e-13
theta =  90.0  M_y: max|M_new - M_stored|/max|M_stored| = 1.61e-14
theta =  90.0  M_z: max|M_new - M_stored|/max|M_stored| = 1.43e-14
theta =   0.0  M_x: max|M_new - M_stored|/max|M_stored| = 2.06e-14
theta =   0.0  M_y: max|M_new - M_stored|/max|M_stored| = 1.96e-14
theta =   0.0  M_z: max|M_new - M_stored|/max|M_stored| = 2.71e-15
theta =  83.2  M_x: max|M_new - M_stored|/max|M_stored| = 6.54e-13
theta =  83.2  M_y: max|M_new - M_stored|/max|M_stored| = 1.23e-14
theta =  83.2  M_z: max|M_new - M_stored|/max|M_stored| = 1.14e-14
--- EDSR/import_data ---
theta =  90.0  M_x: max|M_new - M_stored|/max|M_stored| = 1.16e+00
theta =  90.0  M_y: max|M_new - M_stored|/max|M_stored| = 5.68e-05
theta =  90.0  M_z: max|M_new - M_stored|/max|M_stored| = 1.85e-03
theta =   0.0  M_x: max|M_new - M_stored|/max|M_stored| = 1.81e+00
theta =   0.0  M_y: max|M_new - M_stored|/max|M_stored| = 2.06e-03
theta =   0.0  

## Summary: g-tensor and signs of $g_{ii}$ (|E| = 6 V/µm)

In [8]:
rows = []
print(f"{'direction':>14s} {'g_xx':>9s} {'g_yy':>9s} {'g_zz':>9s} {'sgn(xx,yy,zz)':>14s} "
      f"{'sgn det G':>9s} {'det G':>10s} {'max offdiag/|g|':>16s} {'M∝σ':>5s}")
for th in THETAS_DEG:
    M, G, info = results[th]
    lab = {90.0: "E || y", 0.0: "E || z"}.get(th, f"th={th}")
    d = np.diag(G)
    off = max(abs(G[i, j]) / np.linalg.norm(G[i]) for i in range(3) for j in range(3) if i != j)
    sg = "".join("+" if x > 0 else "-" for x in d)
    detG = np.linalg.det(G)
    print(f"{lab:>14s} {d[0]:+9.4f} {d[1]:+9.4f} {d[2]:+9.4f} {sg:>14s} {'+' if detG > 0 else '-':>9s} "
          f"{detG:+10.4f} {off:16.2e} {'yes' if summary_prop[th] else 'no':>5s}")
    rows.append([th, *G.ravel(), detG])

out = f"g_tensor_E{E_V_um:.1f}Vum_N{N}_L{int(L)}.dat"
np.savetxt(out, np.array(rows),
           header="theta(deg)  gxx gxy gxz  gyx gyy gyz  gzx gzy gzz  detG   "
                  "[G[a,b] = Tr(M_a sigma_b)/muB; basis: psi_new_basis]")
print(f"\nSaved {out}")
print("\nFull g-tensors G[alpha, beta]:")
for th in THETAS_DEG:
    print(f"theta = {th}:\n{results[th][1]}")

     direction      g_xx      g_yy      g_zz  sgn(xx,yy,zz) sgn det G      det G  max offdiag/|g|   M∝σ
        E || y   -0.1940   -3.3199  +13.6252            --+         +    +8.7761         5.11e-13   yes
        E || z   +0.1344   +0.3667  +11.0438            +++         +    +0.5442         3.01e-14   yes
       th=83.2   +0.0659   -0.6341  +17.4754            +-+         -    -1.1869         9.95e-01    no

Saved g_tensor_E6.0Vum_N100_L300.dat

Full g-tensors G[alpha, beta]:
theta = 90.0:
[[-0.19401 -0.      -0.     ]
 [-0.      -3.31993 -0.     ]
 [ 0.      -0.      13.62524]]
theta = 0.0:
[[ 0.13438  0.      -0.     ]
 [ 0.       0.36667 -0.     ]
 [-0.      -0.      11.04379]]
theta = 83.2:
[[ 0.06589 -0.       0.     ]
 [-0.      -0.63411 -6.57485]
 [ 0.      -1.05428 17.47536]]


### Notes on the signs
* In the `psi_new_basis` convention, $\operatorname{sign} g_{zz}$ is set by the ordering (column 0 = lower $\langle S_z\rangle$), and $\operatorname{sign} g_{xx}$ by $\langle1|S_x|2\rangle > 0$. Both are conventions. A $\pi$ pseudospin rotation about one axis flips the other two signs.
* $\operatorname{sign}\det G$ does not depend on the basis. It is the sign that can be measured (e.g. through the direction of Larmor precession or the Berry phase). When $G$ is diagonal it equals $\operatorname{sign}(g_{xx}g_{yy}g_{zz})$.